# 01 · Download datasets
Raw datasets are downloaded to the Colab VM disk (`/content/raw`, large but **wiped when the session ends**).
Only manifests and processed data (notebook 02) are saved to Google Drive.
So run **01 → 02 in the same session**.

| Dataset | Size | Access |
|---|---|---|
| DFDC train sample | ~4 GB download | Kaggle account + API token |
| ASVspoof 2019 LA | 7.6 GB | free |
| FakeAVCeleb | ~ several GB | request form → link by email |
| Celeb-DF v2 | ~ 10 GB | request form → link by email |

In [ ]:
# --- Session setup: run this first, every time you open the notebook ---
REPO_URL = "https://github.com/tanishq1101/robust-multimodal-deepfake.git"  # "" = use code copied to Drive

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

if REPO_URL:
    CODE = "/content/DeepFake"
    if os.path.exists(CODE):
        subprocess.run(["git", "-C", CODE, "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, CODE], check=True)
else:
    # No GitHub repo yet: upload the project folder to Google Drive at MyDrive/deepfake_project/code
    CODE = "/content/drive/MyDrive/deepfake_project/code"
    assert os.path.exists(CODE), f"Upload the project folder to {CODE} or set REPO_URL"

os.chdir(CODE)
!pip install -q -r requirements-colab.txt && pip install -q -e .
sys.path.insert(0, f"{CODE}/src")

from deepfake.config import load_config
cfg = load_config()
print("data_root (Drive):", cfg["data_root"])
print("raw data (VM disk):", cfg["paths"]["raw"])

## 1. DFDC sample (Kaggle)
1. Log in at kaggle.com and **accept the rules**: https://www.kaggle.com/competitions/deepfake-detection-challenge/rules
2. Kaggle → *Settings* → *API* → **Create New Token** → downloads `kaggle.json`.
3. Run the next cell and upload `kaggle.json` when asked.

In [ ]:
from google.colab import files
import os, shutil
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    os.chdir("/content")       # keep the token off Google Drive
    files.upload()             # choose kaggle.json
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    shutil.move("/content/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chdir(CODE)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle token ready")

In [ ]:
!python -m deepfake.data.download dfdc

## 2. ASVspoof 2019 LA (free, ~10–20 min)

In [ ]:
!python -m deepfake.data.download asvspoof

## 3. FakeAVCeleb (after your request is approved)
Paste the link from the approval email below. If they sent a `download.sh` script instead,
upload it with the second cell.

In [ ]:
FAKEAVCELEB_URL = ""  # paste link from the email
if FAKEAVCELEB_URL:
    !python -m deepfake.data.download fakeavceleb --url "$FAKEAVCELEB_URL"

In [ ]:
# Alternative: run the download script they emailed you
# os.chdir("/content"); from google.colab import files; files.upload()   # upload download.sh
# !mkdir -p /content/raw/fakeavceleb && cd /content/raw/fakeavceleb && bash /content/download.sh
# os.chdir(CODE)

## 4. Celeb-DF v2 (after approval)

In [ ]:
CELEBDF_URL = ""  # paste link from the email
if CELEBDF_URL:
    !python -m deepfake.data.download celebdf --url "$CELEBDF_URL"

## 5. Check, then build manifests + identity-wise splits (saved to Drive)

In [ ]:
!python -m deepfake.data.download status

In [ ]:
!python -m deepfake.data.manifest all
import os
for name in ["dfdc", "asvspoof", "fakeavceleb", "celebdf"]:
    if os.path.exists(f"{cfg['paths']['manifests']}/{name}.csv"):
        print(f"--- {name}")
        !python -m deepfake.data.splits {name}